# UNM ds003490 - Preprocesamiento EEG

Este cuaderno implementa el preprocesamiento de las señales EEG en reposo del conjunto
ds003490 (OpenNeuro) utilizado en la tesis. Es el primer paso del pipeline: sus salidas
(épocas limpias por sesión y `epochs_manifest.csv`) son la entrada de los cuadernos 02A y 02B.

La cohorte candidata está formada exclusivamente por **25 HC** y **25 PD en estado OFF**;
los registros PD_ON no se procesan. Tras el control de calidad, la muestra final queda en
**24 HC** y **24 PD_OFF** (verificado en la sección 26).

El flujo aplica las siguientes decisiones metodológicas:

- referencia promedio común;
- filtro FIR de 1 a 40 Hz;
- remuestreo a 250 Hz;
- corrección de artefactos oculares mediante ICA antes de restringir canales;
- selección final de **10 canales**: C3, Cz, C4, P7, P8, P3, Pz, P4, O1 y O2;
- épocas no solapadas de 4 s;
- rechazo por PTP > 100 µV o |z de curtosis| > 5;
- inclusión de sesiones con al menos 6 épocas válidas.

Además, genera los resultados descriptivos del preprocesamiento, la prueba inferencial
sobre componentes ICA, la PSD por regiones sobre los canales seleccionados y la estimación
de la frecuencia del pico alfa (PAF).

## 1. Instalación


In [ ]:
# !pip -q install mne awscli  # instalar con requirements.txt

## 2. Importaciones, semilla y versiones


In [ ]:
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass, field
import logging
import platform
import re
import shutil
import subprocess
import warnings

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import mne
import scipy

warnings.filterwarnings("default")
mne.set_log_level("WARNING")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("eeg_preprocesamiento_final")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Versiones del entorno")
print("Python:", platform.python_version())
print("MNE:", mne.__version__)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("pandas:", pd.__version__)
print("Matplotlib:", matplotlib.__version__)


## 3. Selección de canales del modelo

La representación final utiliza diez derivaciones centrales, parietales y occipitales.
La selección se aplica **después** de la corrección mediante ICA.


In [ ]:
# Diez canales centrales, parietales y occipitales utilizados en el análisis final.
SELECTED_CHANNELS: Tuple[str, ...] = (
    "C3", "Cz", "C4",
    "P7", "P8", "P3", "Pz", "P4",
    "O1", "O2",
)

print(f"Canales seleccionados ({len(SELECTED_CHANNELS)}):", SELECTED_CHANNELS)
assert len(SELECTED_CHANNELS) == 10


## 4. Configuración centralizada (`PreprocessConfig`)


In [ ]:
@dataclass(frozen=True)
class PreprocessConfig:
    """Configuración reproducible del preprocesamiento EEG."""

    selected_channels: Tuple[str, ...] = ()
    non_eeg_channel_types: Dict[str, str] = field(default_factory=lambda: {
        "VEOG": "eog", "HEOG": "eog", "X": "misc", "Y": "misc", "Z": "misc",
    })
    filter_l_freq: float = 1.0
    filter_h_freq: float = 40.0
    target_sfreq: float = 250.0
    segment_mode: str = "EC"
    fallback_duration: float = 60.0
    ica_random_state: int = 42
    ica_max_components_to_exclude: int = 2
    eog_proxy_channels: Tuple[str, ...] = ("Fp1", "Fp2", "AF3", "AF4", "F7", "F8")
    epoch_length_sec: float = 4.0
    epoch_overlap_sec: float = 0.0
    epoch_reject_ptp_uv: float = 100.0
    epoch_reject_kurt_z: float = 5.0
    min_epochs_required: int = 6
    random_state: int = 42


config = PreprocessConfig(selected_channels=SELECTED_CHANNELS)
print(config)


## 5. Rutas y descarga del conjunto de datos

El conjunto ds003490 se descarga desde el repositorio público de OpenNeuro (AWS S3, sin
credenciales). Las salidas se guardan en Google Drive bajo `FINAL_ROOT`; para usar otra
ubicación basta con modificar esa variable en la celda de rutas.

In [ ]:
# Rutas portátiles: Google Drive solo se monta si USE_GOOGLE_DRIVE=1 (Colab).
import os
from pathlib import Path

if os.environ.get('USE_GOOGLE_DRIVE') == '1':
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
else:
    print('Ejecución local: se usan PARKINSON_EEG_ROOT y PARKINSON_EEG_DATA.')

In [ ]:
import os  # rutas portátiles (ver docs/reproducibility.md)
_REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
BASE_DIR = Path(os.environ.get('PARKINSON_EEG_DATA', _REPO_ROOT / 'data' / 'openneuro')).expanduser().resolve()
BASE_DIR.mkdir(parents=True, exist_ok=True)
UNM_PATH = BASE_DIR / "ds003490"

# Carpeta raíz de salidas: defina PARKINSON_EEG_ROOT para usar otra ubicación.
FINAL_ROOT = Path(os.environ.get('PARKINSON_EEG_ROOT', _REPO_ROOT / 'artifacts')).expanduser().resolve()
OUTPUT_DIR = FINAL_ROOT / "01_preprocesamiento"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUTPUT_DIR / "figures_qc"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("UNM_PATH   :", UNM_PATH)
print("FINAL_ROOT :", FINAL_ROOT)
print("OUTPUT_DIR :", OUTPUT_DIR)


In [ ]:
def download_openneuro_s3(dataset_id, target_dir, force_clean=False):
    target_dir = Path(target_dir)
    if force_clean and target_dir.exists():
        shutil.rmtree(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    cmd = ['aws', 's3', 'sync', '--no-sign-request',
           f's3://openneuro.org/{dataset_id}', str(target_dir),
           '--exclude', '*',
           '--include', '*.set', '--include', '*.fdt',
           '--include', '*.json', '--include', '*.tsv',
           '--include', 'participants.tsv', '--include', 'README']
    subprocess.run(cmd, check=True)

download_openneuro_s3('ds003490', UNM_PATH, force_clean=False)

## 6. Índice de sujetos y selección de la cohorte HC/PD_OFF


In [ ]:
def map_condition(row: pd.Series, session: str) -> str:
    """Mapea la sesión original a HC/PD_ON/PD_OFF según participants.tsv."""
    group = str(row["Group"]).upper()
    if group == "CTL":
        return "CTL"
    if group == "PD":
        key = "sess1_Med" if session == "01" else "sess2_Med"
        return f"PD_{row[key]}"
    return "UNKNOWN"


def build_index(unm_path: Path, participants: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for file in sorted(unm_path.rglob("*_eeg.set")):
        subject_match = re.search(r"(sub-\d+)", file.name)
        session_match = re.search(r"ses-(\d+)", file.name)
        if not subject_match or not session_match:
            continue
        subject_id, session = subject_match.group(1), session_match.group(1)
        participant = participants.loc[participants["participant_id"] == subject_id]
        if participant.empty:
            continue
        participant = participant.iloc[0]
        rows.append({
            "subject_id": subject_id,
            "session": session,
            "group": participant["Group"],
            "condition": map_condition(participant, session),
            "set_file": str(file),
        })
    return pd.DataFrame(rows)


participants = pd.read_csv(UNM_PATH / "participants.tsv", sep="\t")
subject_index = build_index(UNM_PATH, participants)

# Cohorte utilizada en la tesis: controles sanos y pacientes en estado OFF.
analysis_index = subject_index[subject_index["condition"].isin(["CTL", "PD_OFF"])].copy()
analysis_index["clase"] = analysis_index["condition"].map({"CTL": "HC", "PD_OFF": "PD_OFF"})

counts = analysis_index.groupby("clase").size()
print("Sesiones candidatas para la corrida final:")
display(counts.rename("n").reset_index())

assert int(counts.get("HC", 0)) == 25, "Se esperaban 25 sesiones HC."
assert int(counts.get("PD_OFF", 0)) == 25, "Se esperaban 25 sesiones PD_OFF."
assert analysis_index["subject_id"].nunique() == 50, "La cohorte candidata debe contener 50 sujetos únicos."


In [ ]:
ex_ctl = analysis_index[analysis_index["condition"] == "CTL"].iloc[0]
ex_pd_off = analysis_index[analysis_index["condition"] == "PD_OFF"].iloc[0]
print("HC    :", ex_ctl["subject_id"])
print("PD_OFF:", ex_pd_off["subject_id"])


## 7. Funciones de visualización

Las visualizaciones de control de calidad no forman parte de la entrada de los modelos.
Las etiquetas de las figuras se presentan en español y se utiliza **HC** como
denominación de los controles sanos.

In [ ]:
INSPECT_CH = ['Fp1','F3','Fz','F4','C3','Cz','C4','P3','Pz','P4','O1','O2']
# Canales para espectro QC: uno frontal, uno central, uno occipital.
# Fp1 se mantiene como referencia visual aunque no entre al modelo.
CH_QC = {'Fp1': '#D85A30', 'Cz': '#534AB7', 'O1': '#0F6E56'}


def plot_timeseries(raw, channels, duration=10., start=5., title='', save_path=None):
    avail = [c for c in channels if c in raw.ch_names]
    if not avail: return
    sr = raw.info['sfreq']
    d, t = raw.get_data(picks=avail,
                        start=int(start*sr), stop=int((start+duration)*sr),
                        return_times=True)
    d = (d - np.median(d, axis=1, keepdims=True)) * 1e6
    step = np.percentile(np.abs(d), 95) * 3 or 1.
    fig, ax = plt.subplots(figsize=(11, 1.4*len(avail)+1))
    for i, ch in enumerate(avail):
        ax.plot(t, d[i] + i*step, lw=0.8)
        ax.text(t[0], i*step, ch, fontsize=8, va='center', ha='right')
    ax.set_yticks([]); ax.set_xlabel('Tiempo (s)'); ax.set_title(title)
    fig.tight_layout()
    if save_path: fig.savefig(save_path, dpi=150)
    plt.show()


def plot_spectrum(raw, ch_colors, fmin=0.5, fmax=100., title='', save_path=None):
    """Espectro QC hasta 100 Hz para visualizar atenuación del ruido de 60 Hz."""
    avail = {c: col for c, col in ch_colors.items() if c in raw.ch_names}
    if not avail: return
    sp = raw.compute_psd(method='welch', fmin=fmin, fmax=fmax,
                         n_fft=int(raw.info['sfreq']*4),
                         n_overlap=int(raw.info['sfreq']*2),
                         picks=list(avail), verbose='ERROR')
    psd, freqs = sp.get_data(return_freqs=True)
    psd_db = 10*np.log10(psd + 1e-20)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for i, (c, col) in enumerate(avail.items()):
        ax.plot(freqs, psd_db[i], label=c, color=col, lw=1.2)
    # Marcadores de referencia
    ax.axvline(60, color='gray', ls='--', lw=.9, alpha=.7, label='60 Hz (ruido de línea)')
    ax.axvspan(8, 13, color='gray', alpha=.08, label='alfa 8-13 Hz')
    ax.axvspan(4, 8,  color='steelblue', alpha=.05, label='theta 4-8 Hz')
    ax.set_xlabel('Frecuencia (Hz)'); ax.set_ylabel('PSD (dB)')
    ax.set_title(title); ax.legend(fontsize=7)
    fig.tight_layout()
    if save_path: fig.savefig(save_path, dpi=150)
    plt.show()


def channel_stats(raw, picks='eeg'):
    rs = raw.copy().pick(picks)
    d  = rs.get_data() * 1e6
    df = pd.DataFrame([{'channel': ch, 'mean_uv': float(np.mean(d[i])),
                         'std_uv': float(np.std(d[i])), 'ptp_uv': float(np.ptp(d[i]))}
                        for i, ch in enumerate(rs.ch_names)])
    df['z']    = (df['ptp_uv'] - df['ptp_uv'].mean()) / (df['ptp_uv'].std() + 1e-8)
    df['flag'] = df['z'].abs() > 2.5
    return df

## 8. Paso 1 - Carga del montaje completo y corrección de tipos de canal

La corrección mediante ICA se realiza sobre el montaje EEG completo. La restricción
a los diez canales de análisis ocurre posteriormente.


In [ ]:
def fix_ch_types(raw, ch_map):
    r = raw.copy()
    r.set_channel_types({c: t for c, t in ch_map.items() if c in r.ch_names},
                         verbose='ERROR')
    return r


def find_events_file(set_file):
    f = Path(set_file).with_name(Path(set_file).name.replace('_eeg.set', '_events.tsv'))
    return f if f.exists() else None


def crop_segment(raw, ev_file, mode, fallback):
    evts = pd.read_csv(ev_file, sep='\t')
    if 'onset' not in evts.columns or 'trial_type' not in evts.columns:
        return raw.copy().crop(0, min(fallback, raw.times[-1]))
    label = 'Eyes Closed' if mode == 'EC' else 'Eyes Open'
    sel = evts[evts['trial_type'].astype(str).str.contains(label, case=False, na=False)]
    if sel.empty:
        return raw.copy().crop(0, min(fallback, raw.times[-1]))
    return raw.copy().crop(float(sel['onset'].min()),
                           min(float(sel['onset'].max()) + 1., raw.times[-1]))


def load_full_montage(set_file, config):
    raw = mne.io.read_raw_eeglab(str(set_file), preload=True, verbose='ERROR')
    raw = fix_ch_types(raw, config.non_eeg_channel_types)
    ev  = find_events_file(set_file)
    if ev:
        raw = crop_segment(raw, ev, config.segment_mode, config.fallback_duration)
    else:
        raw = raw.copy().crop(0, min(config.fallback_duration, raw.times[-1]))
    return raw


raw_ctl    = load_full_montage(ex_ctl['set_file'],    config)
raw_pd_off = load_full_montage(ex_pd_off['set_file'], config)

for lbl, raw in [('HC', raw_ctl), ('PD_OFF', raw_pd_off)]:
    tipos = pd.Series(raw.get_channel_types()).value_counts()
    print(f'{lbl} | canales={len(raw.ch_names)} | dur={raw.times[-1]:.1f}s')
    print(tipos.to_string(), '\n')

## 9. Señal cruda - visualización de control


In [ ]:
plot_timeseries(raw_ctl,    INSPECT_CH, title='HC — crudo',
                save_path=FIG_DIR/'ctl_01_crudo.png')
plot_timeseries(raw_pd_off, INSPECT_CH, title='PD_OFF — crudo',
                save_path=FIG_DIR/'pd_off_01_crudo.png')

# Espectro hasta 100 Hz: debe verse pico de 60 Hz antes de filtrar
plot_spectrum(raw_ctl,    CH_QC, title='HC — espectro crudo (hasta 100 Hz)',
              save_path=FIG_DIR/'ctl_01_espectro.png')
plot_spectrum(raw_pd_off, CH_QC, title='PD_OFF — espectro crudo (hasta 100 Hz)',
              save_path=FIG_DIR/'pd_off_01_espectro.png')

st0_ctl    = channel_stats(raw_ctl)
st0_pd_off = channel_stats(raw_pd_off)
print('HC     outliers crudo:', st0_ctl[st0_ctl['flag']]['channel'].tolist())
print('PD_OFF outliers crudo:', st0_pd_off[st0_pd_off['flag']]['channel'].tolist())

## 10. Paso 2 - Referencia promedio


In [ ]:
def apply_ref(raw):
    r = raw.copy()
    r.set_eeg_reference('average', projection=False, verbose='ERROR')
    return r

raw_ctl_ref    = apply_ref(raw_ctl)
raw_pd_off_ref = apply_ref(raw_pd_off)
print('Referencia promedio aplicada.')

## 11. Paso 3 - Filtro de 1 a 40 Hz y remuestreo a 250 Hz

El espectro de control se muestra hasta 100 Hz para verificar la atenuación del contenido
de 60 Hz y el corte superior del filtro.


In [ ]:
def apply_filter_resample(raw, config):
    r = raw.copy()
    picks = mne.pick_types(r.info, eeg=True, eog=True, misc=False)
    r.filter(config.filter_l_freq, config.filter_h_freq,
             picks=picks, fir_design='firwin', verbose='ERROR')
    r.resample(config.target_sfreq, verbose='ERROR')
    return r

raw_ctl_filt    = apply_filter_resample(raw_ctl_ref,    config)
raw_pd_off_filt = apply_filter_resample(raw_pd_off_ref, config)

plot_timeseries(raw_ctl_filt,    INSPECT_CH,
                title='HC — tras referencia+filtro+resample',
                save_path=FIG_DIR/'ctl_02_filtrado.png')
plot_timeseries(raw_pd_off_filt, INSPECT_CH,
                title='PD_OFF — tras referencia+filtro+resample',
                save_path=FIG_DIR/'pd_off_02_filtrado.png')

# Post-filtro: el pico de 60 Hz debe haber desaparecido
plot_spectrum(raw_ctl_filt,    CH_QC,
              title='HC — espectro post-filtro (60 Hz debe estar eliminado)',
              save_path=FIG_DIR/'ctl_02_espectro.png')
plot_spectrum(raw_pd_off_filt, CH_QC,
              title='PD_OFF — espectro post-filtro (60 Hz debe estar eliminado)',
              save_path=FIG_DIR/'pd_off_02_espectro.png')

## 12. Paso 4 - Corrección de artefactos mediante ICA

Se utiliza FastICA con un máximo de 20 componentes y se excluyen como máximo dos
componentes asociados con actividad ocular por sesión.


In [ ]:
def apply_ica(raw, config):
    r = raw.copy()
    if len(r.ch_names) < 4:
        return r, None, [], {'eog_source': None, 'excluded': []}

    real_eog = mne.pick_types(r.info, eog=True)
    use_real = len(real_eog) > 0

    n_comp = min(20, len(mne.pick_types(r.info, eeg=True)) - 1)
    ica = mne.preprocessing.ICA(n_components=n_comp, method='fastica',
                                  random_state=config.ica_random_state,
                                  max_iter='auto', verbose='ERROR')
    ica.fit(r, picks='eeg', verbose='ERROR')

    excl = []
    if use_real:
        eog_names = [r.ch_names[i] for i in real_eog]
        try:
            inds, _ = ica.find_bads_eog(r, ch_name=eog_names,
                                         threshold=3.0, verbose='ERROR')
            excl.extend(inds)
        except Exception as e:
            logger.warning('find_bads_eog real fallo: %s', e)
            use_real = False
    if not use_real:
        for p in [c for c in config.eog_proxy_channels if c in r.ch_names]:
            try:
                inds, _ = ica.find_bads_eog(r, ch_name=p,
                                              threshold=3.0, verbose='ERROR')
                excl.extend(inds)
            except Exception:
                pass

    excl = list(dict.fromkeys(excl))[:config.ica_max_components_to_exclude]
    ica.exclude = excl
    r = ica.apply(r, verbose='ERROR')

    return r, ica, excl, {'eog_source': 'real_eog' if use_real else 'frontal_proxy',
                           'excluded': excl}


raw_ctl_ica,    _, excl_ctl,    rep_ctl    = apply_ica(raw_ctl_filt,    config)
raw_pd_off_ica, _, excl_pd_off, rep_pd_off = apply_ica(raw_pd_off_filt, config)

print('HC     | eog_source:', rep_ctl['eog_source'],    '| excluidos:', excl_ctl)
print('PD_OFF | eog_source:', rep_pd_off['eog_source'], '| excluidos:', excl_pd_off)

## 13. Comparación visual después de ICA


In [ ]:
plot_timeseries(raw_ctl_ica,    INSPECT_CH,
                title='HC — tras ICA (montaje completo)',
                save_path=FIG_DIR/'ctl_03_ica.png')
plot_timeseries(raw_pd_off_ica, INSPECT_CH,
                title='PD_OFF — tras ICA (montaje completo)',
                save_path=FIG_DIR/'pd_off_03_ica.png')

plot_spectrum(raw_ctl_ica,    CH_QC,
              title='HC — espectro post-ICA',
              save_path=FIG_DIR/'ctl_03_espectro.png')
plot_spectrum(raw_pd_off_ica, CH_QC,
              title='PD_OFF — espectro post-ICA',
              save_path=FIG_DIR/'pd_off_03_espectro.png')

st3_ctl    = channel_stats(raw_ctl_ica)
st3_pd_off = channel_stats(raw_pd_off_ica)
print('HC     outliers post-ICA:', st3_ctl[st3_ctl['flag']]['channel'].tolist())
print('PD_OFF outliers post-ICA:', st3_pd_off[st3_pd_off['flag']]['channel'].tolist())

## 14. Paso 5 - Restricción a los 10 canales del análisis

La reducción del montaje se realiza únicamente después de ICA.


In [ ]:
def restrict_channels(raw, config):
    final = [c for c in config.selected_channels if c in raw.ch_names]
    return raw.copy().pick(final)

raw_ctl_final    = restrict_channels(raw_ctl_ica,    config)
raw_pd_off_final = restrict_channels(raw_pd_off_ica, config)
print('HC     canales finales:', raw_ctl_final.ch_names)
print('PD_OFF canales finales:', raw_pd_off_final.ch_names)

## 15. Paso 6 - Segmentación en épocas fijas de 4 s


In [ ]:
def epoch_signal(raw, config):
    return mne.make_fixed_length_epochs(
        raw, duration=config.epoch_length_sec,
        overlap=config.epoch_overlap_sec,
        preload=True, verbose='ERROR')

epochs_ctl    = epoch_signal(raw_ctl_final,    config)
epochs_pd_off = epoch_signal(raw_pd_off_final, config)
print('HC     épocas:', len(epochs_ctl))
print('PD_OFF epocas:', len(epochs_pd_off))

## 16. Paso 7 - Rechazo multicriterio de épocas

Una época se rechaza cuando, en cualquiera de los diez canales seleccionados, presenta
PTP > 100 µV o |z de curtosis| > 5. Los criterios se aplican sobre la señal ya corregida
mediante ICA.


In [ ]:
from scipy.stats import kurtosis as scipy_kurtosis

REJECT_PTP_UV = config.epoch_reject_ptp_uv
REJECT_KURT_Z = config.epoch_reject_kurt_z


def reject_epochs_multicriteria(
    epochs: mne.Epochs,
    ptp_threshold_uv: float = REJECT_PTP_UV,
    kurt_z_threshold: float = REJECT_KURT_Z,
):
    """Rechaza épocas por PTP absoluto y/o curtosis anómala."""
    picks = mne.pick_types(epochs.info, eeg=True)
    ch_names = [epochs.ch_names[i] for i in picks]
    data_uv = epochs.get_data(picks=picks) * 1e6

    ptp = np.ptp(data_uv, axis=2)
    flag_ptp = ptp > ptp_threshold_uv

    kurt = np.array([
        [scipy_kurtosis(data_uv[ep, ch, :], fisher=True)
         for ch in range(data_uv.shape[1])]
        for ep in range(data_uv.shape[0])
    ])
    kurt_mean = kurt.mean(axis=0, keepdims=True)
    kurt_std = kurt.std(axis=0, keepdims=True)
    kurt_std = np.where(kurt_std < 1e-8, 1.0, kurt_std)
    kurt_z = (kurt - kurt_mean) / kurt_std
    flag_kurt = np.abs(kurt_z) > kurt_z_threshold

    keep_mask = ~(flag_ptp | flag_kurt).any(axis=1)

    report = pd.DataFrame({
        "epoch_idx": np.arange(len(keep_mask)),
        "kept": keep_mask,
        "max_ptp_uv": ptp.max(axis=1),
        "max_kurt_z": np.abs(kurt_z).max(axis=1),
        "flag_ptp": flag_ptp.any(axis=1),
        "flag_kurt": flag_kurt.any(axis=1),
        "ch_ptp": [
            ch_names[ptp[ep].argmax()] if flag_ptp[ep].any() else ""
            for ep in range(len(keep_mask))
        ],
        "ch_kurt": [
            ch_names[np.abs(kurt_z[ep]).argmax()] if flag_kurt[ep].any() else ""
            for ep in range(len(keep_mask))
        ],
    })

    n_total = len(keep_mask)
    n_kept = int(keep_mask.sum())
    fraction = n_kept / max(n_total, 1)
    logger.info("Épocas conservadas: %d/%d (%.0f%%)", n_kept, n_total, fraction * 100)
    return keep_mask, report


In [ ]:
keep_ctl,    rep_ep_ctl    = reject_epochs_multicriteria(epochs_ctl)
keep_pd_off, rep_ep_pd_off = reject_epochs_multicriteria(epochs_pd_off)

print(f'HC     conservadas: {keep_ctl.sum()}/{len(keep_ctl)}'
      f' ({100*keep_ctl.mean():.0f}%)')
print(f'PD_OFF conservadas: {keep_pd_off.sum()}/{len(keep_pd_off)}'
      f' ({100*keep_pd_off.mean():.0f}%)')

for lbl, rep in [('HC', rep_ep_ctl), ('PD_OFF', rep_ep_pd_off)]:
    rechaz = rep[~rep['kept']]
    if rechaz.empty:
        print(f'{lbl}: sin epocas rechazadas')
    else:
        print(f'\n{lbl} — epocas rechazadas:')
        print(rechaz[['epoch_idx','max_ptp_uv','flag_ptp','ch_ptp',
                       'max_kurt_z','flag_kurt','ch_kurt']].to_string(index=False))

In [ ]:
def plot_epoch_rejection(report: pd.DataFrame, label: str = "", save_path: Optional[Path] = None) -> None:
    """Visualiza PTP y curtosis por época para control de calidad."""
    n_epochs = len(report)
    n_kept = int(report["kept"].sum())
    colors = ["#1D9E75" if kept else "#D85A30" for kept in report["kept"]]
    fig, axes = plt.subplots(2, 1, figsize=(max(8, n_epochs * 0.5), 7), sharex=True)

    axes[0].bar(report["epoch_idx"], report["max_ptp_uv"], color=colors)
    axes[0].axhline(REJECT_PTP_UV, color="crimson", ls="--", lw=1.2,
                    label=f"Umbral PTP ({REJECT_PTP_UV:.0f} µV)")
    axes[0].set_ylabel("PTP máximo (µV)")
    axes[0].legend(fontsize=8)
    axes[0].set_title(f"Rechazo de épocas - {label}: {n_kept}/{n_epochs} conservadas")

    axes[1].bar(report["epoch_idx"], report["max_kurt_z"], color=colors)
    axes[1].axhline(REJECT_KURT_Z, color="darkorange", ls="--", lw=1.2,
                    label=f"Umbral |z de curtosis| ({REJECT_KURT_Z:.0f})")
    axes[1].set_xlabel("Índice de época")
    axes[1].set_ylabel("|z de curtosis| máximo")
    axes[1].legend(fontsize=8)

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


plot_epoch_rejection(rep_ep_ctl, "HC", save_path=FIG_DIR / "hc_04_rechazo_epocas.png")
plot_epoch_rejection(rep_ep_pd_off, "PD_OFF", save_path=FIG_DIR / "pd_off_04_rechazo_epocas.png")


## 17. Resumen de los dos sujetos utilizados para control visual


In [ ]:
resumen = pd.DataFrame([
    {'sujeto': ex_ctl['subject_id'],    'condicion': 'HC',
     'eog_source': rep_ctl['eog_source'],
     'componentes_excluidos': str(excl_ctl),
     'n_epocas_total': len(keep_ctl),
     'n_epocas_conservadas': int(keep_ctl.sum()),
     'fraccion_conservada': round(float(keep_ctl.mean()), 2)},
    {'sujeto': ex_pd_off['subject_id'], 'condicion': 'PD_OFF',
     'eog_source': rep_pd_off['eog_source'],
     'componentes_excluidos': str(excl_pd_off),
     'n_epocas_total': len(keep_pd_off),
     'n_epocas_conservadas': int(keep_pd_off.sum()),
     'fraccion_conservada': round(float(keep_pd_off.mean()), 2)},
])
display(resumen)
resumen.to_csv(OUTPUT_DIR / 'resumen_preprocesamiento_ejemplos.csv', index=False)

## 18. Verificaciones visuales del preprocesamiento

Las figuras de las secciones anteriores permiten comprobar, en los sujetos de ejemplo, que:

- los canales EOG están disponibles y se emplean en la identificación de componentes oculares;
- el espectro crudo muestra el contenido de 60 Hz de la red eléctrica;
- después del filtrado, el contenido por encima de 40 Hz queda atenuado;
- las trazas posteriores a ICA no presentan artefactos oculares dominantes;
- el número de épocas conservadas es compatible con los criterios de inclusión.

## 19. Pipeline completo y guardado de resultados

El pipeline procesa las **50 sesiones candidatas** (25 HC y 25 PD_OFF). Genera
`epochs_manifest.csv` y un archivo `.npy` por sesión con las épocas válidas de los diez
canales seleccionados.

In [ ]:
RUN_ALL = True  # True: ejecuta el pipeline completo sobre las 50 sesiones candidatas.

if RUN_ALL:
    from tqdm.auto import tqdm

    EPOCHS_DIR = OUTPUT_DIR / "preprocessed_epochs"
    EPOCHS_DIR.mkdir(parents=True, exist_ok=True)
    manifest_rows = []

    for _, row in tqdm(analysis_index.iterrows(), total=len(analysis_index),
                       desc="Preprocesando HC y PD_OFF"):
        subject_id = row["subject_id"]
        condition = row["condition"]
        try:
            raw_full = load_full_montage(row["set_file"], config)
            raw_ref = apply_ref(raw_full)
            raw_filt = apply_filter_resample(raw_ref, config)
            raw_ica, _, excluded, ica_report = apply_ica(raw_filt, config)
            raw_final = restrict_channels(raw_ica, config)
            epochs = epoch_signal(raw_final, config)
            keep_mask, epoch_report = reject_epochs_multicriteria(epochs)

            epochs_kept = epochs[keep_mask]
            data_uv = epochs_kept.get_data() * 1e6

            picks = mne.pick_types(epochs.info, eeg=True)
            channel_names = [epochs.ch_names[i] for i in picks]
            data_all_uv = epochs.get_data(picks=picks) * 1e6
            ptp_mean = np.ptp(data_all_uv, axis=2).mean(axis=0)
            noisiest_channel = channel_names[int(ptp_mean.argmax())]

            file_name = f"{subject_id}_{condition}.npy"
            file_path = EPOCHS_DIR / file_name
            np.save(file_path, data_uv.astype(np.float32))

            manifest_rows.append({
                "subject_id": subject_id,
                "condition": condition,
                "group": row["group"],
                "session": row["session"],
                "file": str(file_path),
                "eog_source": ica_report["eog_source"],
                "n_comp_excluidos": len(excluded),
                "comp_excluidos": str(excluded),
                "n_epocas_total": len(keep_mask),
                "n_epocas_conservadas": int(keep_mask.sum()),
                "fraccion_conservada": float(keep_mask.mean()),
                "max_ptp_uv": float(epoch_report["max_ptp_uv"].max()),
                "n_flag_ptp": int(epoch_report["flag_ptp"].sum()),
                "n_flag_kurt": int(epoch_report["flag_kurt"].sum()),
                "canal_mas_ruidoso": noisiest_channel,
                "n_channels": data_uv.shape[1],
                "n_times": data_uv.shape[2],
                "sfreq": float(config.target_sfreq),
                "ch_names": ",".join(epochs_kept.ch_names),
                "error": "",
            })

        except Exception as exc:
            logger.exception("Error en %s %s", subject_id, condition)
            manifest_rows.append({
                "subject_id": subject_id, "condition": condition,
                "group": row.get("group", ""), "session": row.get("session", ""),
                "file": "", "eog_source": "",
                "n_comp_excluidos": -1, "comp_excluidos": "",
                "n_epocas_total": 0, "n_epocas_conservadas": 0,
                "fraccion_conservada": 0.0, "max_ptp_uv": np.nan,
                "n_flag_ptp": 0, "n_flag_kurt": 0,
                "canal_mas_ruidoso": "",
                "n_channels": 0, "n_times": 0, "sfreq": 0.0,
                "ch_names": "", "error": str(exc),
            })

    manifest = pd.DataFrame(manifest_rows)
    manifest_path = OUTPUT_DIR / "epochs_manifest.csv"
    manifest.to_csv(manifest_path, index=False)

    print("\n" + "=" * 72)
    print("PREPROCESAMIENTO COMPLETO")
    print("=" * 72)
    print("Manifest:", manifest_path)
    print("Épocas:", EPOCHS_DIR)

    ok = manifest[manifest["error"].fillna("") == ""].copy()
    print(f"Sesiones procesadas sin error: {len(ok)}/{len(manifest)}")
    display(ok.groupby("condition").agg(
        n_sesiones=("subject_id", "count"),
        epocas_media=("n_epocas_conservadas", "mean"),
        epocas_min=("n_epocas_conservadas", "min"),
        fraccion_media=("fraccion_conservada", "mean"),
        n_componentes_ica_media=("n_comp_excluidos", "mean"),
    ).round(3))

    if not manifest[manifest["error"].fillna("") != ""].empty:
        print("\nSesiones con error:")
        display(manifest.loc[manifest["error"].fillna("") != "",
                             ["subject_id", "condition", "error"]])
else:
    manifest_path = OUTPUT_DIR / "epochs_manifest.csv"
    if not manifest_path.exists():
        raise FileNotFoundError("No existe un manifest previo. Ejecute con RUN_ALL=True.")
    manifest = pd.read_csv(manifest_path)
    print(f"Manifest cargado: {len(manifest)} sesiones")


## 20. Estadística descriptiva del preprocesamiento e inferencia sobre ICA

Se resumen las épocas generadas y conservadas, las tasas de marcado por PTP y curtosis
y el número de componentes ICA excluidos. Estos resultados se reportan en tablas de la tesis.

La comparación HC vs PD_OFF del número de componentes ICA se calcula con la prueba U de
Mann-Whitney y tamaño de efecto rank-biserial. Esta comparación describe la salida del
procedimiento de corrección y no debe interpretarse como una diferencia clínica en
cantidad de artefactos oculares.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

DESC_DIR = OUTPUT_DIR / "descriptiva_preproc"
DESC_DIR.mkdir(parents=True, exist_ok=True)
MIN_EPOCHS = config.min_epochs_required

_man = pd.read_csv(OUTPUT_DIR / "epochs_manifest.csv")
if "error" in _man.columns:
    _man["error"] = _man["error"].fillna("").astype(str)
    _man = _man[_man["error"] == ""]
_man = _man[_man["condition"].isin(["CTL", "PD_OFF"])].copy()
_man["clase"] = _man["condition"].map({"CTL": "HC", "PD_OFF": "PD_OFF"})


def preprocessing_quality_table(m: pd.DataFrame, min_epochs: int = MIN_EPOCHS) -> pd.DataFrame:
    """Resume calidad del preprocesamiento para HC, PD_OFF y total."""
    rows = []
    for group in ["HC", "PD_OFF", "Total"]:
        subset = m if group == "Total" else m[m["clase"] == group]
        included = subset[subset["n_epocas_conservadas"] >= min_epochs]
        kept = included["n_epocas_conservadas"].astype(float)
        total_epochs = float(subset["n_epocas_total"].sum())
        rows.append({
            "Grupo": group,
            "Sesiones candidatas": len(subset),
            "Sesiones incluidas": len(included),
            "Sesiones excluidas": len(subset) - len(included),
            "Épocas generadas": int(total_epochs),
            "Épocas conservadas (media ± DE)": (
                f"{kept.mean():.1f} ± {kept.std():.1f}" if len(kept) else "n/a"
            ),
            "Conservación media por sesión (%)": (
                round(100 * included["fraccion_conservada"].mean(), 1) if len(included) else np.nan
            ),
            "Marcadas por PTP (%)": round(100 * subset["n_flag_ptp"].sum() / max(total_epochs, 1), 1),
            "Marcadas por curtosis (%)": round(100 * subset["n_flag_kurt"].sum() / max(total_epochs, 1), 1),
            "Componentes ICA (media ± DE)": (
                f"{subset['n_comp_excluidos'].mean():.2f} ± {subset['n_comp_excluidos'].std():.2f}"
            ),
        })
    return pd.DataFrame(rows)


qtab = preprocessing_quality_table(_man)
print("=" * 72)
print("CALIDAD DEL PREPROCESAMIENTO")
print("=" * 72)
display(qtab)
qtab.to_csv(DESC_DIR / "descriptiva_preprocesamiento.csv", index=False)

# Comparación del número de épocas conservadas entre los 48 sujetos incluidos.
included = _man[_man["n_epocas_conservadas"] >= MIN_EPOCHS].copy()
hc_epochs = included[included["clase"] == "HC"]["n_epocas_conservadas"].to_numpy()
pd_epochs = included[included["clase"] == "PD_OFF"]["n_epocas_conservadas"].to_numpy()
u_epochs, p_epochs = mannwhitneyu(hc_epochs, pd_epochs, alternative="two-sided", method="asymptotic")
print(f"\nÉpocas conservadas HC vs PD_OFF: U={u_epochs:.1f}, p={p_epochs:.4f}")
print("Interpretación: contraste descriptivo de retención; no se usa para seleccionar sujetos.")

# Comparación inferencial de componentes ICA sobre las 50 sesiones candidatas.
hc_ica = _man[_man["clase"] == "HC"]["n_comp_excluidos"].astype(float).to_numpy()
pd_ica = _man[_man["clase"] == "PD_OFF"]["n_comp_excluidos"].astype(float).to_numpy()
u_ica, p_ica = mannwhitneyu(hc_ica, pd_ica, alternative="two-sided", method="asymptotic")
r_rb_ica = 1.0 - (2.0 * u_ica) / (len(hc_ica) * len(pd_ica))

ica_summary = pd.DataFrame({
    "Grupo": ["HC", "PD_OFF", "Total"],
    "n": [len(hc_ica), len(pd_ica), len(hc_ica) + len(pd_ica)],
    "Media": [hc_ica.mean(), pd_ica.mean(), np.r_[hc_ica, pd_ica].mean()],
    "DE": [hc_ica.std(ddof=1), pd_ica.std(ddof=1), np.r_[hc_ica, pd_ica].std(ddof=1)],
})
ica_test = pd.DataFrame([{
    "U": float(u_ica),
    "p_bilateral": float(p_ica),
    "rank_biserial": float(r_rb_ica),
    "abs_rank_biserial": float(abs(r_rb_ica)),
    "n_HC": len(hc_ica),
    "n_PD_OFF": len(pd_ica),
}])

print("\nComponentes ICA excluidos por sesión:")
display(ica_summary.round(3))
print(f"Mann-Whitney HC vs PD_OFF: U={u_ica:.1f}, p={p_ica:.6f}, "
      f"r_rb={r_rb_ica:.3f}, |r_rb|={abs(r_rb_ica):.3f}")
print("Nota: este resultado describe la salida del procedimiento ICA; no demuestra una diferencia clínica de artefactos.")

ica_summary.to_csv(DESC_DIR / "ica_resumen_por_grupo.csv", index=False)
ica_test.to_csv(DESC_DIR / "ica_estadistica.csv", index=False)
_man[["subject_id", "condition", "clase", "session", "n_comp_excluidos", "comp_excluidos"]].to_csv(
    DESC_DIR / "ica_por_sesion.csv", index=False
)


## 21. PSD relativa por región en los 10 canales seleccionados

La caracterización regional se construye exclusivamente con los diez canales utilizados
en los análisis posteriores. Para mejorar la legibilidad, se definen cuatro agrupamientos
operativos: **central** (C3, Cz, C4), **parietal lateral** (P7, P8), **parietal medial**
(P3, Pz, P4) y **occipital** (O1, O2).

Cada curva corresponde a la media regional calculada primero a nivel de sujeto; las
bandas sombreadas representan intervalos bootstrap del 95 % entre sujetos. La figura se
organiza en una cuadrícula 2 x 2 y utiliza las bandas theta (4-8 Hz) y alfa (8-13 Hz)
como referencias visuales del intervalo de mayor interés neurofisiológico.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import welch
from scipy.integrate import simpson

DESC_DIR = OUTPUT_DIR / "descriptiva_preproc"
DESC_DIR.mkdir(parents=True, exist_ok=True)

PSD_FMIN, PSD_FMAX = 1.0, 30.0
WELCH_SEC, WELCH_OVERLAP = 2.0, 0.5
GROUP_COLORS = {"HC": "#2C7FB8", "PD_OFF": "#D7301F"}
REGIONS_10 = {
    "Central": ["C3", "Cz", "C4"],
    "Parietal lateral": ["P7", "P8"],
    "Parietal medial": ["P3", "Pz", "P4"],
    "Occipital": ["O1", "O2"],
}


def relative_log_psd(
    epochs_uv: np.ndarray,
    sfreq: float,
    ch_names: list[str],
) -> tuple[np.ndarray, np.ndarray]:
    """Calcula log10(PSD relativa 1-30 Hz) y promedia las épocas por sujeto."""
    nperseg = min(int(sfreq * WELCH_SEC), epochs_uv.shape[-1])
    noverlap = int(nperseg * WELCH_OVERLAP)
    freqs, psd = welch(
        epochs_uv,
        fs=sfreq,
        nperseg=nperseg,
        noverlap=noverlap,
        axis=-1,
    )
    mask = (freqs >= PSD_FMIN) & (freqs <= PSD_FMAX)
    freqs_band = freqs[mask]
    psd_band = psd[..., mask]
    total = simpson(psd_band, x=freqs_band, axis=-1)
    total = np.maximum(total, 1e-20)
    rel = psd_band / total[..., None]
    return np.log10(rel + 1e-20).mean(axis=0), freqs_band


def bootstrap_ci(
    curves: np.ndarray,
    n_boot: int = 1000,
    seed: int = RANDOM_STATE,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Media e intervalo bootstrap percentil del 95 % a nivel de sujeto."""
    rng = np.random.default_rng(seed)
    n_subjects = len(curves)
    boot = np.empty((n_boot, curves.shape[1]), dtype=float)
    for b in range(n_boot):
        idx = rng.integers(0, n_subjects, n_subjects)
        boot[b] = curves[idx].mean(axis=0)
    return (
        curves.mean(axis=0),
        np.percentile(boot, 2.5, axis=0),
        np.percentile(boot, 97.5, axis=0),
    )


man = pd.read_csv(OUTPUT_DIR / "epochs_manifest.csv")
man["error"] = man.get("error", "").fillna("").astype(str)
man = man[
    (man["error"] == "")
    & (man["n_epocas_conservadas"] >= config.min_epochs_required)
    & (man["condition"].isin(["CTL", "PD_OFF"]))
].copy()
man["clase"] = man["condition"].map({"CTL": "HC", "PD_OFF": "PD_OFF"})

subject_psd = {"HC": [], "PD_OFF": []}
freqs_ref = None
for _, row in man.iterrows():
    npy_path = Path(row["file"])
    if not npy_path.exists():
        raise FileNotFoundError(f"No existe el archivo esperado: {npy_path}")
    data = np.load(npy_path).astype(np.float64)
    ch_names = str(row["ch_names"]).split(",")
    curves, freqs = relative_log_psd(data, float(row["sfreq"]), ch_names)
    channel_index = {ch: i for i, ch in enumerate(ch_names)}
    missing = [ch for ch in SELECTED_CHANNELS if ch not in channel_index]
    if missing:
        raise ValueError(f"Faltan canales en {row['subject_id']}: {missing}")
    ordered = np.stack([curves[channel_index[ch]] for ch in SELECTED_CHANNELS], axis=0)
    subject_psd[row["clase"]].append(ordered)
    freqs_ref = freqs

subject_psd = {group: np.stack(values, axis=0) for group, values in subject_psd.items()}
assert subject_psd["HC"].shape[0] == 24
assert subject_psd["PD_OFF"].shape[0] == 24

region_curves = {group: {} for group in ["HC", "PD_OFF"]}
selected_list = list(SELECTED_CHANNELS)
for group in ["HC", "PD_OFF"]:
    for region, channels in REGIONS_10.items():
        idx = [selected_list.index(ch) for ch in channels]
        region_curves[group][region] = subject_psd[group][:, idx, :].mean(axis=1)

fig, axes = plt.subplots(2, 2, figsize=(11.8, 7.6), sharex=True, sharey=True)
axes = axes.ravel()
for ax, (region, channels) in zip(axes, REGIONS_10.items()):
    for group in ["HC", "PD_OFF"]:
        mean, low, high = bootstrap_ci(region_curves[group][region])
        ax.plot(freqs_ref, mean, lw=2.0, color=GROUP_COLORS[group], label=group)
        ax.fill_between(freqs_ref, low, high, color=GROUP_COLORS[group], alpha=0.16)

    ax.axvspan(4, 8, color="#DBEAFE", alpha=0.30)
    ax.axvspan(8, 13, color="#DCFCE7", alpha=0.30)
    ax.axvline(8, color="0.45", ls="--", lw=0.8, alpha=0.7)
    ax.text(6.0, 0.97, "Theta", transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=8, color="0.35")
    ax.text(10.5, 0.97, "Alfa", transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=8, color="0.35")
    ax.set_title(f"{region} ({', '.join(channels)})", fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.12, linewidth=0.5)

fig.supxlabel("Frecuencia (Hz)", fontsize=10)
fig.supylabel(r"$\log_{10}$(PSD relativa)", fontsize=10)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles[:2], labels[:2], loc="upper center", ncol=2,
           frameon=False, fontsize=9, bbox_to_anchor=(0.5, 1.01))
fig.tight_layout(rect=(0.02, 0.02, 1.0, 0.96))

fig_path = DESC_DIR / "psd_relativa_por_region.png"
fig.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print("Figura guardada:", fig_path)


## 22. Agrupamiento de los 10 canales por región

Esquema auxiliar que documenta de forma explícita el agrupamiento de canales empleado
en la figura regional.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

DESC_DIR = OUTPUT_DIR / "descriptiva_preproc"
DESC_DIR.mkdir(parents=True, exist_ok=True)

POSITIONS = {
    "C3": (-0.45, 0.0), "Cz": (0.0, 0.0), "C4": (0.45, 0.0),
    "P7": (-0.70, -0.50), "P3": (-0.35, -0.52), "Pz": (0.0, -0.55),
    "P4": (0.35, -0.52), "P8": (0.70, -0.50),
    "O1": (-0.30, -0.90), "O2": (0.30, -0.90),
}
REGION_COLORS = {
    "Central": "#4C72B0",
    "Parietal lateral": "#55A868",
    "Parietal medial": "#C44E52",
    "Occipital": "#8172B3",
}
channel_to_region = {ch: region for region, channels in REGIONS_10.items() for ch in channels}

fig, ax = plt.subplots(figsize=(6.2, 6.2))
ax.add_patch(Circle((0, 0), 1.06, fill=False, lw=2, color="0.3"))
ax.plot([0, -0.09, 0.09, 0], [1.06, 1.22, 1.22, 1.06], color="0.3", lw=2)
for channel, (x, y) in POSITIONS.items():
    region = channel_to_region[channel]
    ax.scatter(x, y, s=560, color=REGION_COLORS[region], edgecolor="white", zorder=3)
    ax.text(x, y, channel, ha="center", va="center", fontsize=8,
            color="white", fontweight="bold", zorder=4)
handles = [plt.Line2D([0], [0], marker="o", ls="", markersize=10,
                      markerfacecolor=color, markeredgecolor="white", label=region)
           for region, color in REGION_COLORS.items()]
ax.legend(handles=handles, loc="upper right", frameon=True, fontsize=9)
ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-1.3, 1.35)
ax.set_aspect("equal")
ax.axis("off")
fig.tight_layout()
fig.savefig(DESC_DIR / "agrupamiento_10_canales.png", dpi=250, bbox_inches="tight")
plt.show()


## 22.1. Consistencia de canales

Se verifica que todos los registros incluidos contengan exactamente los diez canales
seleccionados y en el mismo orden lógico esperado.


In [ ]:
man_check = pd.read_csv(OUTPUT_DIR / "epochs_manifest.csv")
man_check["error"] = man_check.get("error", "").fillna("").astype(str)
man_check = man_check[(man_check["error"] == "") &
                      (man_check["n_epocas_conservadas"] >= config.min_epochs_required)]
expected_channels = list(SELECTED_CHANNELS)
for _, row in man_check.iterrows():
    observed = str(row["ch_names"]).split(",")
    if observed != expected_channels:
        raise ValueError(f"Orden de canales inesperado en {row['subject_id']}: {observed}")
print(f"Canales verificados en {len(man_check)} sesiones incluidas: {expected_channels}")


## 23. Efecto del preprocesamiento en el dominio temporal y espectral

Se generan dos versiones de la comparación temporal:

1. una **versión con cuatro canales representativos** (Cz, Pz, O1 y P3), que corresponde
   a la figura incluida en la tesis;
2. una **versión compacta en Pz**, complementaria, que no se incluye en el documento.

Ambas representan exactamente los mismos sujetos de ejemplo y el mismo flujo de
preprocesamiento. Las figuras espectrales muestran por separado el dominio espectral
para HC y PD_OFF en Pz, con una leyenda explícita en español para las señales, las
bandas theta/alfa y las frecuencias de referencia de 40 y 60 Hz.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

DESC_DIR = OUTPUT_DIR / "descriptiva_preproc"
DESC_DIR.mkdir(parents=True, exist_ok=True)

TIME_CHANNELS = ["Cz", "Pz", "O1", "P3"]
TIME_CHANNEL_SINGLE = "Pz"
SPEC_CHANNEL = "Pz"
PLOT_DURATION = 10.0
PLOT_START = 5.0
COLOR_RAW = "#8A8A8A"
COLOR_PROCESSED = {"HC": "#1D9E75", "PD_OFF": "#D85A30"}


def get_window_multi(
    raw: mne.io.BaseRaw,
    channels: list[str],
    start: float,
    duration: float,
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """Extrae una ventana temporal y centra cada canal por su mediana."""
    available = [ch for ch in channels if ch in raw.ch_names]
    sfreq = raw.info["sfreq"]
    data, times = raw.get_data(
        picks=available,
        start=int(start * sfreq),
        stop=int((start + duration) * sfreq),
        return_times=True,
    )
    centered_uv = (data - np.median(data, axis=1, keepdims=True)) * 1e6
    return centered_uv, times, available


def get_window(
    raw: mne.io.BaseRaw,
    channel: str,
    start: float,
    duration: float,
) -> tuple[np.ndarray, np.ndarray]:
    data, times, channels = get_window_multi(raw, [channel], start, duration)
    if channel not in channels:
        raise ValueError(f"Canal {channel} no disponible.")
    return data[channels.index(channel)], times


def get_psd_db(
    raw: mne.io.BaseRaw,
    channel: str,
    fmin: float = 0.5,
    fmax: float = 80.0,
) -> tuple[np.ndarray, np.ndarray]:
    """PSD de visualización para comparar la señal antes y después."""
    if channel not in raw.ch_names:
        raise ValueError(f"Canal {channel} no disponible.")
    fmax_safe = min(fmax, raw.info["sfreq"] / 2.0 - 0.5)
    spectrum = raw.compute_psd(
        method="welch",
        fmin=fmin,
        fmax=fmax_safe,
        n_fft=int(raw.info["sfreq"] * 4),
        n_overlap=int(raw.info["sfreq"] * 2),
        picks=[channel],
        verbose="ERROR",
    )
    psd, freqs = spectrum.get_data(return_freqs=True)
    return 10 * np.log10(psd[0] + 1e-20), freqs


subjects = [
    ("HC", raw_ctl, raw_ctl_final, ex_ctl["subject_id"]),
    ("PD_OFF", raw_pd_off, raw_pd_off_final, ex_pd_off["subject_id"]),
]

# Comparación temporal: cuatro canales representativos x dos sujetos (figura de la tesis).
fig, axes = plt.subplots(
    len(TIME_CHANNELS), 2,
    figsize=(13.5, 9.6),
    sharex="col",
)
for col, (label, raw_before, raw_after, subject_id) in enumerate(subjects):
    before, time_before, before_channels = get_window_multi(
        raw_before, TIME_CHANNELS, PLOT_START, PLOT_DURATION
    )
    after, time_after, after_channels = get_window_multi(
        raw_after, TIME_CHANNELS, PLOT_START, PLOT_DURATION
    )
    for row, channel in enumerate(TIME_CHANNELS):
        ax = axes[row, col]
        if channel in before_channels:
            ax.plot(
                time_before,
                before[before_channels.index(channel)],
                color=COLOR_RAW,
                lw=0.9,
                ls="--",
                alpha=0.8,
                label="Señal cruda",
            )
        if channel in after_channels:
            ax.plot(
                time_after,
                after[after_channels.index(channel)],
                color=COLOR_PROCESSED[label],
                lw=1.1,
                alpha=0.95,
                label="Señal preprocesada",
            )
        ax.set_ylabel(f"{channel}\nAmplitud (µV)", fontsize=8.5)
        ax.spines[["top", "right"]].set_visible(False)
        ax.tick_params(labelsize=8)
        if row == 0:
            ax.set_title(f"{label} - {subject_id}", fontsize=10)
            ax.legend(frameon=False, fontsize=8, loc="upper right")
        if row == len(TIME_CHANNELS) - 1:
            ax.set_xlabel("Tiempo (s)")

fig.tight_layout()
path_full = DESC_DIR / "comparacion_temporal_4canales.png"
fig.savefig(path_full, dpi=300, bbox_inches="tight")
plt.show()
print("Figura guardada:", path_full)

# Versión complementaria compacta en Pz (no incluida en la tesis).
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.1), sharey=True)
for ax, (label, raw_before, raw_after, subject_id) in zip(axes, subjects):
    before, time_before = get_window(
        raw_before, TIME_CHANNEL_SINGLE, PLOT_START, PLOT_DURATION
    )
    after, time_after = get_window(
        raw_after, TIME_CHANNEL_SINGLE, PLOT_START, PLOT_DURATION
    )
    ax.plot(
        time_before, before,
        color=COLOR_RAW, lw=0.9, ls="--", alpha=0.8,
        label="Señal cruda",
    )
    ax.plot(
        time_after, after,
        color=COLOR_PROCESSED[label], lw=1.1, alpha=0.95,
        label="Señal preprocesada",
    )
    ax.set_title(f"{label} - {subject_id} - canal {TIME_CHANNEL_SINGLE}", fontsize=10)
    ax.set_xlabel("Tiempo (s)")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc="upper right")
axes[0].set_ylabel("Amplitud (µV)")
fig.tight_layout()
path_pz = DESC_DIR / "comparacion_temporal_Pz.png"
fig.savefig(path_pz, dpi=300, bbox_inches="tight")
plt.show()
print("Figura guardada:", path_pz)

# Comparación espectral independiente por grupo, canal Pz.
for label, raw_before, raw_after, subject_id in subjects:
    psd_before, freq_before = get_psd_db(raw_before, SPEC_CHANNEL)
    psd_after, freq_after = get_psd_db(raw_after, SPEC_CHANNEL)

    fig, ax = plt.subplots(figsize=(7.4, 4.7))
    ax.plot(
        freq_before, psd_before,
        color=COLOR_RAW, lw=1.3, ls="--", alpha=0.9,
        label="Señal cruda",
    )
    ax.plot(
        freq_after, psd_after,
        color=COLOR_PROCESSED[label], lw=1.8,
        label="Señal preprocesada",
    )
    ax.axvspan(4, 8, color="#DBEAFE", alpha=0.50, label="Theta (4-8 Hz)")
    ax.axvspan(8, 13, color="#DCFCE7", alpha=0.50, label="Alfa (8-13 Hz)")
    ax.axvline(60, color="#6B7280", ls=":", lw=1.2, label="60 Hz (ruido de línea)")
    ax.axvline(40, color="#9CA3AF", ls=":", lw=0.9, label="40 Hz (corte del filtro)")

    ax.set_title(f"{label} - {subject_id} - canal {SPEC_CHANNEL}", fontsize=10)
    ax.set_xlabel("Frecuencia (Hz)")
    ax.set_ylabel("PSD (dB)")
    ax.legend(loc="upper right", fontsize=7.5, frameon=True, framealpha=0.90)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()

    filename = DESC_DIR / f"espectro_{label}_Pz.png"
    fig.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()
    print("Figura guardada:", filename)


## 24. Tabla auxiliar de la muestra final

Resume los sujetos **incluidos después del control de calidad** y el número de épocas
conservadas por sujeto. No sustituye la tabla de caracterización de la cohorte de la tesis, cuyas características
demográficas corresponden a la cohorte original de 25 HC y 25 PD.

In [ ]:
# ============================================================================
# 24. TABLA CLÍNICA AUXILIAR DE LA MUESTRA FINAL
#     Solo sujetos incluidos tras control de calidad (24 HC + 24 PD_OFF).
# ============================================================================
import pandas as pd

DESC_DIR = OUTPUT_DIR / "descriptiva_preproc"
DESC_DIR.mkdir(parents=True, exist_ok=True)


def _find_col(df: pd.DataFrame, *candidates: str) -> str | None:
    lookup = {column.lower(): column for column in df.columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    return None


def enriched_clinical_table(
    participants: pd.DataFrame,
    manifest_path: Path,
    min_epochs: int,
) -> pd.DataFrame:
    """Resume la muestra final incluida; no representa la cohorte demográfica original."""
    col_id = _find_col(participants, "participant_id", "participant", "id")
    col_updrs = _find_col(participants, "UPDRS", "updrs", "UPDRS_total")
    col_mmse = _find_col(participants, "MMSE", "mmse")

    man = pd.read_csv(manifest_path)
    man["error"] = man.get("error", "").fillna("").astype(str)
    man = man[
        (man["error"] == "")
        & (man["n_epocas_conservadas"] >= min_epochs)
        & (man["condition"].isin(["CTL", "PD_OFF"]))
    ].copy()
    man["grupo"] = man["condition"].map({"CTL": "HC", "PD_OFF": "PD_OFF"})

    used = man[["subject_id", "grupo"]].drop_duplicates("subject_id")
    if col_id is not None:
        demo = used.merge(
            participants,
            left_on="subject_id",
            right_on=col_id,
            how="left",
            suffixes=("", "_participants"),
        )
    else:
        demo = used.copy()

    epochs_by_group = (
        man.groupby("grupo")["n_epocas_conservadas"]
        .agg(["mean", "std", "sum"])
        .to_dict("index")
    )

    rows = []
    for group in ["HC", "PD_OFF"]:
        subset = demo[demo["grupo"] == group]
        row = {"Grupo": group, "n final": len(subset)}

        if group == "PD_OFF" and col_updrs and pd.api.types.is_numeric_dtype(demo[col_updrs]):
            values = subset[col_updrs].dropna()
            row["UPDRS (media ± DE)"] = (
                f"{values.mean():.1f} ± {values.std():.1f}" if len(values) else "n/a"
            )
        else:
            row["UPDRS (media ± DE)"] = "N/A (HC)" if group == "HC" else "n/a"

        if col_mmse and pd.api.types.is_numeric_dtype(demo[col_mmse]):
            values = subset[col_mmse].dropna()
            row["MMSE (media ± DE)"] = (
                f"{values.mean():.1f} ± {values.std():.1f}" if len(values) else "n/a"
            )

        epochs = epochs_by_group[group]
        row["Épocas conservadas (media ± DE)"] = f"{epochs['mean']:.1f} ± {epochs['std']:.1f}"
        row["Épocas conservadas (total)"] = int(epochs["sum"])
        rows.append(row)

    return pd.DataFrame(rows)


clin = enriched_clinical_table(
    participants=participants,
    manifest_path=OUTPUT_DIR / "epochs_manifest.csv",
    min_epochs=config.min_epochs_required,
)

print("=" * 72)
print("TABLA CLÍNICA AUXILIAR - MUESTRA FINAL")
print("=" * 72)
display(clin)

assert clin.set_index("Grupo").loc["HC", "n final"] == 24
assert clin.set_index("Grupo").loc["PD_OFF", "n final"] == 24

clinical_path = DESC_DIR / "tabla_clinica_muestra_final.csv"
clin.to_csv(clinical_path, index=False)
print("Guardada:", clinical_path)


## 25. Frecuencia del pico alfa (PAF): HC vs PD_OFF

La PAF se estima en los canales posteriores O1, O2, Pz, P3 y P4 dentro de 7-13 Hz.
Se reportan dos estimadores: **centro de gravedad** (estimación principal) y **máximo
espectral** (estimación complementaria). La figura presenta los valores individuales,
la distribución por grupo y las medianas fuera del área de datos para evitar solapamientos.

El análisis caracteriza la cohorte y no forma parte del clasificador.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import welch
from scipy.stats import mannwhitneyu

DESC_DIR = OUTPUT_DIR / "descriptiva_preproc"
DESC_DIR.mkdir(parents=True, exist_ok=True)

PAF_FMIN, PAF_FMAX = 7.0, 13.0
PAF_CHANNELS = ["O1", "O2", "Pz", "P3", "P4"]


def compute_paf_subject(
    epochs_uv: np.ndarray,
    sfreq: float,
    ch_names: list[str],
    wanted_channels: list[str],
) -> dict[str, float]:
    """Calcula PAF mediante centro de gravedad y máximo espectral."""
    nperseg = min(int(sfreq * 2.0), epochs_uv.shape[-1])
    noverlap = int(nperseg * 0.5)
    freqs, psd = welch(epochs_uv, fs=sfreq, nperseg=nperseg, noverlap=noverlap, axis=-1)
    psd_subject = psd.mean(axis=0)
    idx = [ch_names.index(ch) for ch in wanted_channels if ch in ch_names]
    if not idx:
        return {"cog": np.nan, "argmax": np.nan}
    posterior = psd_subject[idx].mean(axis=0)
    band = (freqs >= PAF_FMIN) & (freqs <= PAF_FMAX)
    freq_band = freqs[band]
    power_band = posterior[band]
    if power_band.sum() <= 0:
        return {"cog": np.nan, "argmax": np.nan}
    return {
        "cog": float(np.sum(freq_band * power_band) / np.sum(power_band)),
        "argmax": float(freq_band[np.argmax(power_band)]),
    }


def rank_biserial(x: np.ndarray, y: np.ndarray, u: float) -> float:
    """Correlación rank-biserial r = 1 - 2U / (n_x * n_y), con U el estadístico de Mann-Whitney asociado a x.

    r > 0 indica que los valores de y tienden a ser mayores que los de x.
    """
    return float(1.0 - (2.0 * u) / (len(x) * len(y)))


man = pd.read_csv(OUTPUT_DIR / "epochs_manifest.csv")
man["error"] = man.get("error", "").fillna("").astype(str)
man = man[(man["error"] == "") &
          (man["n_epocas_conservadas"] >= config.min_epochs_required) &
          (man["condition"].isin(["CTL", "PD_OFF"]))].copy()
man["clase"] = man["condition"].map({"CTL": "HC", "PD_OFF": "PD_OFF"})

# Las rutas se toman del manifest generado en este cuaderno para evitar cualquier mezcla PD_ON/PD_OFF.
if man["subject_id"].duplicated().any():
    duplicates = man.loc[man["subject_id"].duplicated(keep=False), ["subject_id", "condition", "file"]]
    raise ValueError(f"Se encontraron sujetos duplicados en el manifest final:\n{duplicates}")

records = []
for _, row in man.iterrows():
    npy_path = Path(row["file"])
    if not npy_path.exists():
        raise FileNotFoundError(f"No existe el archivo esperado: {npy_path}")
    data = np.load(npy_path).astype(np.float64)
    ch_names = str(row["ch_names"]).split(",")
    paf = compute_paf_subject(data, float(row["sfreq"]), ch_names, PAF_CHANNELS)
    records.append({
        "subject_id": row["subject_id"],
        "clase": row["clase"],
        "PAF_cog": paf["cog"],
        "PAF_argmax": paf["argmax"],
    })

paf_df = pd.DataFrame(records).dropna(subset=["PAF_cog"])
paf_df.to_csv(DESC_DIR / "paf_por_sujeto.csv", index=False)
print(f"PAF calculada para {len(paf_df)} sujetos: "
      f"HC={(paf_df['clase'] == 'HC').sum()}, PD_OFF={(paf_df['clase'] == 'PD_OFF').sum()}")


def compare_groups(df: pd.DataFrame, column: str) -> dict[str, float]:
    hc = df.loc[df["clase"] == "HC", column].dropna().to_numpy()
    pd_off = df.loc[df["clase"] == "PD_OFF", column].dropna().to_numpy()
    u, p_two = mannwhitneyu(hc, pd_off, alternative="two-sided", method="asymptotic")
    _, p_directional = mannwhitneyu(pd_off, hc, alternative="less", method="asymptotic")
    return {
        "HC_mediana": float(np.median(hc)),
        "PD_OFF_mediana": float(np.median(pd_off)),
        "diferencia_Hz": float(np.median(hc) - np.median(pd_off)),
        "U": float(u),
        "p_bilateral": float(p_two),
        "p_PD_OFF_menor_HC": float(p_directional),
        "rank_biserial": rank_biserial(hc, pd_off, u),
    }


results = {
    "PAF_cog": compare_groups(paf_df, "PAF_cog"),
    "PAF_argmax": compare_groups(paf_df, "PAF_argmax"),
}
print("=" * 72)
print("FRECUENCIA DEL PICO ALFA - HC vs PD_OFF")
print("=" * 72)
for key, label in [("PAF_cog", "Centro de gravedad (principal)"),
                   ("PAF_argmax", "Máximo espectral (complementario)")]:
    print(f"\n{label}")
    for name, value in results[key].items():
        print(f"  {name:24s}: {value:.6f}" if isinstance(value, float) else f"  {name}: {value}")

pd.DataFrame(results).T.to_csv(DESC_DIR / "paf_estadistica.csv")

# Figura de PAF por grupo
fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.8), sharey=True)
panels = [
    ("PAF_cog", "Centro de gravedad (estimación principal)"),
    ("PAF_argmax", "Máximo espectral (estimación complementaria)"),
]

for ax, (column, title) in zip(axes, panels):
    hc = paf_df.loc[paf_df["clase"] == "HC", column].to_numpy()
    pd_off = paf_df.loc[paf_df["clase"] == "PD_OFF", column].to_numpy()

    box = ax.boxplot(
        [hc, pd_off],
        tick_labels=["HC", "PD_OFF"],
        patch_artist=True,
        widths=0.58,
        showmeans=True,
    )
    for patch, color in zip(box["boxes"], ["#2C7FB8", "#D7301F"]):
        patch.set_facecolor(color)
        patch.set_alpha(0.35)

    rng = np.random.default_rng(RANDOM_STATE)
    for i, (values, color) in enumerate(
        [(hc, "#2C7FB8"), (pd_off, "#D7301F")],
        start=1,
    ):
        ax.scatter(
            rng.normal(i, 0.055, len(values)),
            values,
            color=color,
            s=26,
            alpha=0.72,
            zorder=3,
        )

    ax.set_title(title, fontsize=10, pad=34)
    ax.text(
        0.5,
        1.025,
        f"Medianas: HC = {np.median(hc):.2f} Hz   |   PD_OFF = {np.median(pd_off):.2f} Hz",
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=8.5,
    )
    ax.set_xlabel("Grupo")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.10, linewidth=0.5)

axes[0].set_ylabel("Frecuencia estimada del pico alfa (Hz)")
fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.91))

paf_figure_path = DESC_DIR / "paf_HC_vs_PD_OFF.png"
fig.savefig(paf_figure_path, dpi=300, bbox_inches="tight")
plt.show()
print("Figura guardada:", paf_figure_path)
print("Nota: clasificación experimental y caracterización de biomarcadores; no constituye diagnóstico clínico.")


## 26. Verificación de la cohorte y de las salidas

Esta celda comprueba que la muestra final contenga 24 HC y 24 PD_OFF y que existan los
archivos principales generados por el cuaderno.

In [ ]:
final_manifest = pd.read_csv(OUTPUT_DIR / "epochs_manifest.csv")
final_manifest["error"] = final_manifest.get("error", "").fillna("").astype(str)
final_manifest = final_manifest[(final_manifest["error"] == "") &
                                (final_manifest["n_epocas_conservadas"] >= config.min_epochs_required)].copy()
final_manifest["clase"] = final_manifest["condition"].map({"CTL": "HC", "PD_OFF": "PD_OFF"})

counts_final = final_manifest.groupby("clase")["subject_id"].nunique().to_dict()
print("Cohorte final:", counts_final)
assert counts_final.get("HC", 0) == 24, "La corrida no produjo 24 HC incluidos."
assert counts_final.get("PD_OFF", 0) == 24, "La corrida no produjo 24 PD_OFF incluidos."
assert len(final_manifest) == 48, "La muestra final debe contener 48 sesiones/sujetos."

print("\nArchivos principales generados:")
for path in [
    OUTPUT_DIR / "epochs_manifest.csv",
    OUTPUT_DIR / "descriptiva_preproc" / "descriptiva_preprocesamiento.csv",
    OUTPUT_DIR / "descriptiva_preproc" / "ica_estadistica.csv",
    OUTPUT_DIR / "descriptiva_preproc" / "comparacion_temporal_4canales.png",
    OUTPUT_DIR / "descriptiva_preproc" / "comparacion_temporal_Pz.png",
    OUTPUT_DIR / "descriptiva_preproc" / "espectro_HC_Pz.png",
    OUTPUT_DIR / "descriptiva_preproc" / "espectro_PD_OFF_Pz.png",
    OUTPUT_DIR / "descriptiva_preproc" / "psd_relativa_por_region.png",
    OUTPUT_DIR / "descriptiva_preproc" / "paf_estadistica.csv",
    OUTPUT_DIR / "descriptiva_preproc" / "paf_HC_vs_PD_OFF.png",
    OUTPUT_DIR / "descriptiva_preproc" / "tabla_clinica_muestra_final.csv",
]:
    print(" -", path, "OK" if path.exists() else "FALTA")

print("\nCuaderno 01 verificado: cohorte y archivos principales disponibles.")


## 27. Exportación de datos descriptivos para interpretabilidad funcional

Se exportan los biomarcadores descriptivos a nivel de sujeto para su utilización
posterior en el análisis de interpretabilidad del clasificador (cuaderno 05).

Estos archivos no constituyen entradas adicionales al modelo. Su finalidad es
permitir contrastar las funciones de peso obtenidas posteriormente a partir de la
regresión logística con los patrones espectrales observados en la cohorte.

Se generan:

1. PAF individual mediante centro de gravedad y máximo espectral.
2. PSD relativa logarítmica por sujeto, región y frecuencia.

In [ ]:
# ============================================================
# EXPORTACIÓN PARA INTERPRETABILIDAD
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


INTERP_DIR = OUTPUT_DIR / "descriptiva_preproc" / "interpretabilidad"
INTERP_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Cargar y validar manifest final
# ------------------------------------------------------------

manifest_interp = pd.read_csv(OUTPUT_DIR / "epochs_manifest.csv")

if "error" in manifest_interp.columns:
    manifest_interp["error"] = manifest_interp["error"].fillna("").astype(str)

manifest_interp = manifest_interp[
    (manifest_interp["error"] == "")
    & (
        manifest_interp["n_epocas_conservadas"]
        >= config.min_epochs_required
    )
    & manifest_interp["condition"].isin(["CTL", "PD_OFF"])
].copy()

manifest_interp["grupo"] = manifest_interp["condition"].map({
    "CTL": "HC",
    "PD_OFF": "PD_OFF",
})

manifest_interp = manifest_interp.sort_values(
    ["grupo", "subject_id"]
).reset_index(drop=True)

assert len(manifest_interp) == 48
assert manifest_interp["subject_id"].nunique() == 48

counts = manifest_interp.groupby("grupo")["subject_id"].nunique().to_dict()

assert counts == {
    "HC": 24,
    "PD_OFF": 24,
}, f"Cohorte inesperada: {counts}"


# ============================================================
# 2. PAF POR SUJETO
# ============================================================

paf_records = []

for _, row in manifest_interp.iterrows():

    npy_path = Path(row["file"])

    if not npy_path.exists():
        raise FileNotFoundError(
            f"No existe el archivo de épocas: {npy_path}"
        )

    epochs_uv = np.load(npy_path).astype(np.float64)
    ch_names = str(row["ch_names"]).split(",")

    paf = compute_paf_subject(
        epochs_uv=epochs_uv,
        sfreq=float(row["sfreq"]),
        ch_names=ch_names,
        wanted_channels=PAF_CHANNELS,
    )

    paf_records.append({
        "subject_id": row["subject_id"],
        "grupo": row["grupo"],
        "paf_centro_gravedad_hz": paf["cog"],
        "paf_maximo_espectral_hz": paf["argmax"],
    })


paf_interp = pd.DataFrame(paf_records)

assert len(paf_interp) == 48
assert paf_interp["subject_id"].nunique() == 48
assert not paf_interp[
    ["paf_centro_gravedad_hz", "paf_maximo_espectral_hz"]
].isna().any().any()

paf_path = INTERP_DIR / "paf_por_sujeto.csv"
paf_interp.to_csv(paf_path, index=False)


# ============================================================
# 3. PSD REGIONAL POR SUJETO Y FRECUENCIA
# ============================================================

psd_records = []

selected_channels = list(SELECTED_CHANNELS)

for _, row in manifest_interp.iterrows():

    npy_path = Path(row["file"])
    epochs_uv = np.load(npy_path).astype(np.float64)

    ch_names = str(row["ch_names"]).split(",")

    # Misma función utilizada para construir la figura de PSD relativa por región.
    # Devuelve:
    #   curvas: canales x frecuencias
    #   freqs : frecuencias entre 1 y 30 Hz
    curves, freqs = relative_log_psd(
        epochs_uv=epochs_uv,
        sfreq=float(row["sfreq"]),
        ch_names=ch_names,
    )

    channel_index = {
        ch: i
        for i, ch in enumerate(ch_names)
    }

    missing = [
        ch
        for ch in selected_channels
        if ch not in channel_index
    ]

    if missing:
        raise ValueError(
            f"{row['subject_id']} no contiene los canales: {missing}"
        )

    # --------------------------------------------------------
    # Promedio regional dentro de cada sujeto
    # --------------------------------------------------------

    for region, channels in REGIONS_10.items():

        idx = [
            channel_index[ch]
            for ch in channels
        ]

        regional_curve = curves[idx, :].mean(axis=0)

        for freq, value in zip(freqs, regional_curve):

            psd_records.append({
                "subject_id": row["subject_id"],
                "grupo": row["grupo"],
                "region": region,
                "frecuencia_hz": float(freq),
                "log10_psd_relativa": float(value),
            })


psd_regional_interp = pd.DataFrame(psd_records)

# ------------------------------------------------------------
# Validaciones de integridad
# ------------------------------------------------------------

assert psd_regional_interp["subject_id"].nunique() == 48

assert set(psd_regional_interp["region"].unique()) == set(
    REGIONS_10.keys()
)

n_freqs = psd_regional_interp["frecuencia_hz"].nunique()

expected_rows = (
    48
    * len(REGIONS_10)
    * n_freqs
)

assert len(psd_regional_interp) == expected_rows, (
    f"Filas PSD inesperadas: "
    f"{len(psd_regional_interp)} != {expected_rows}"
)

duplicates = psd_regional_interp.duplicated(
    subset=[
        "subject_id",
        "region",
        "frecuencia_hz",
    ]
).sum()

assert duplicates == 0, (
    f"Se encontraron {duplicates} registros duplicados."
)

psd_path = INTERP_DIR / "psd_regional_por_sujeto.csv"
psd_regional_interp.to_csv(
    psd_path,
    index=False,
)


# ============================================================
# 4. Mapa región-canal
# ============================================================

region_channel_rows = []

for region, channels in REGIONS_10.items():
    for channel in channels:
        region_channel_rows.append({
            "region": region,
            "canal": channel,
        })

region_channel_df = pd.DataFrame(region_channel_rows)

region_channel_path = INTERP_DIR / "mapa_regiones_canales.csv"
region_channel_df.to_csv(
    region_channel_path,
    index=False,
)


# ============================================================
# 5. Resumen
# ============================================================

print("=" * 72)
print("EXPORTACIÓN PARA INTERPRETABILIDAD")
print("=" * 72)

print("\nCohorte:")
print(counts)

print("\nPAF por sujeto:")
print(f"  Filas: {len(paf_interp)}")
print(f"  Sujetos: {paf_interp['subject_id'].nunique()}")
print(f"  Archivo: {paf_path}")

print("\nPSD regional por sujeto:")
print(f"  Filas: {len(psd_regional_interp):,}")
print(
    f"  Frecuencias: "
    f"{psd_regional_interp['frecuencia_hz'].min():.1f}"
    f"–"
    f"{psd_regional_interp['frecuencia_hz'].max():.1f} Hz"
)
print(f"  Número de frecuencias: {n_freqs}")
print(f"  Regiones: {list(REGIONS_10.keys())}")
print(f"  Archivo: {psd_path}")

print("\nMapa de canales:")
print(f"  Archivo: {region_channel_path}")

print("\nExportación para interpretabilidad completada correctamente.")